## Querying PostgreSQL via SDK

### Installing Utilities and Libraries

In [2]:
%pip install psycopg[binary]==3.3.4 psycopg_pool==3.3.1 python-dotenv
%pip install azure-identity psycopg2-binary

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-intel 2.11.0 requires protobuf<3.20,>=3.9.2, but you have protobuf 3.20.3 which is incompatible.

[notice] A new release of pip is available: 24.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



   ---------------------------------------- 0.0/213.0 kB ? eta -:--:--
   ------------- -------------------------- 71.7/213.0 kB 2.0 MB/s eta 0:00:01
   ------------------------------------ --- 194.6/213.0 kB 2.0 MB/s eta 0:00:01
   -------------------------------------- - 204.8/213.0 kB 1.8 MB/s eta 0:00:01
   -------------------------------------- - 204.8/213.0 kB 1.8 MB/s eta 0:00:01
   -------------------------------------- - 204.8/213.0 kB 1.8 MB/s eta 0:00:01
   -------------------------------------- 213.0/213.0 kB 866.5 kB/s eta 0:00:00
   ---------------------------------------- 0.0/40.0 kB ? eta -:--:--
   ------------------------------ --------- 30.7/40.0 kB ? eta -:--:--
   ------------------------------ --------- 30.7/40.0 kB ? eta -:--:--
   ------------------------------ --------- 30.7/40.0 kB ? eta -:--:--
   ------------------------------ --------- 30.7/40.0 kB ? eta -:--:--
   ---------------------------------------- 40.0/40.0 kB 159.3 kB/s eta 0:00:00
   ------------


[notice] A new release of pip is available: 24.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Setting up the Environment

In [ ]:
import os
import psycopg2
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

host = os.getenv("DATABASE_HOSTNAME")
db_name = os.getenv("DATABASE_NAME")
username = os.getenv("DATABASE_USERNAME")

# 1. Acquire Entra ID Access Token dynamically
credential = DefaultAzureCredential()
token = credential.get_token("https://ossrdbms-aad.database.windows.net/.default")
password = token.token

# 2. Connect to PostgreSQL using the dynamic Entra token as the password
conn = psycopg2.connect(
    host=host,
    dbname=db_name,
    user=username,
    password=password,
    port=5432,
    sslmode="require"  # Azure PostgreSQL requires SSL for Entra ID authentication
)

print("Successfully connected using Entra ID!")

Successfully connected using Entra ID!


### Create a Connection Pool

In [ ]:
from psycopg_pool import ConnectionPool

pool = ConnectionPool(
    conninfo=(
        f"host={host} "
        f"dbname={db_name} "
        f"user={username} "
        f"password={password} "
        f"sslmode=require"
    ),
    min_size=2,
    max_size=10
)

pool.wait()

print("Connection pool created successfully")

### Use the Connection Pool to Execute a SELECT Query

In [10]:
import os
import psycopg
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv
from psycopg_pool import ConnectionPool

load_dotenv()

host = os.getenv("DATABASE_HOSTNAME")
db_name = os.getenv("DATABASE_NAME")
username = os.getenv("DATABASE_USERNAME")

# 1. Initialize Azure credential helper
credential = DefaultAzureCredential()

# 2. Define a custom connection class that injects the Entra ID token
class EntraConnection(psycopg.Connection):
    @classmethod
    def connect(cls, conninfo="", **kwargs):
        # Retrieve a fresh token for every new connection establishment
        token = credential.get_token("https://ossrdbms-aad.database.windows.net/.default")
        kwargs["password"] = token.token
        return super().connect(conninfo, **kwargs)

# 3. Connection string without hardcoded password
conninfo = f"host={host} dbname={db_name} user={username} sslmode=require"

# 4. Initialize pool with the custom connection_class
pool = ConnectionPool(
    conninfo=conninfo,
    connection_class=EntraConnection,
    open=True
)

# 5. Query execution using your pool context manager
with pool.connection() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT *
            FROM ESG.Companies
        """)

        results = cur.fetchall()

        for row in results:
            print(row)

# Close pool when application shuts down


(1, 'GreenSteel Ltd', 'Manufacturing', 'IN', datetime.datetime(2026, 9, 28, 12, 11, 12, 959400))
(2, 'EcoLogistics Pvt Ltd', 'Transportation', 'IN', datetime.datetime(2026, 9, 28, 12, 11, 12, 959400))
(3, 'SolarGrid Energy', 'Energy', 'US', datetime.datetime(2026, 9, 28, 12, 11, 12, 959400))
(4, 'UrbanRetail Corp', 'Retail', 'UK', datetime.datetime(2026, 9, 28, 12, 11, 12, 959400))


### Filter Data Using WHERE

In [11]:
with pool.connection() as conn:
    with conn.cursor() as cur:
        cur.execute("""
        SELECT *
        FROM ESG.EmissionRecords
        WHERE Scope = %s
        """,
        (1,))

        results = cur.fetchall()

        for row in results:
            print(row)

(1, 1, datetime.date(2025, 1, 15), 1, Decimal('1200.50'), 'Furnace Combustion')
(3, 2, datetime.date(2025, 1, 10), 1, Decimal('450.25'), 'Fleet Fuel')


error connecting in 'pool-1': connection failed: connection to server at "172.198.76.154", port 5432 failed: fe_sendauth: no password supplied


### Filter Emissions Greater than 500

In [ ]:
with pool.connection() as conn:
    with conn.cursor() as cur:
        cur.execute("""
        SELECT *
        FROM ESG.EmissionRecords
        WHERE CO2_Emissions > %s
        """,
        (500,))

        results = cur.fetchall()

        for row in results:
            print(row)

(1, 1, datetime.date(2025, 1, 15), 1, Decimal('1200.50'), 'Furnace Combustion')
(2, 1, datetime.date(2025, 1, 20), 2, Decimal('800.75'), 'Electricity Usage')
(7, 4, datetime.date(2025, 1, 8), 2, Decimal('600.60'), 'Store Electricity')


error connecting in 'pool-2': connection failed: connection to server at "172.198.76.154", port 5432 failed: fe_sendauth: no password supplied
error connecting in 'pool-2': connection failed: connection to server at "172.198.76.154", port 5432 failed: fe_sendauth: no password supplied


### Sort by Emissions

In [13]:
with pool.connection() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT *
            FROM ESG.EmissionRecords
            ORDER BY CO2_Emissions ASC
        """)

        results = cur.fetchall()

        for row in results:
            print(row)

(6, 3, datetime.date(2025, 1, 18), 3, Decimal('90.00'), 'Supply Chain')
(5, 3, datetime.date(2025, 1, 5), 2, Decimal('150.10'), 'Grid Electricity')
(8, 4, datetime.date(2025, 1, 22), 3, Decimal('250.30'), 'Logistics')
(4, 2, datetime.date(2025, 1, 12), 3, Decimal('300.40'), 'Third-party Transport')
(3, 2, datetime.date(2025, 1, 10), 1, Decimal('450.25'), 'Fleet Fuel')
(7, 4, datetime.date(2025, 1, 8), 2, Decimal('600.60'), 'Store Electricity')
(2, 1, datetime.date(2025, 1, 20), 2, Decimal('800.75'), 'Electricity Usage')
(1, 1, datetime.date(2025, 1, 15), 1, Decimal('1200.50'), 'Furnace Combustion')


### Calculate Total Emissions per Company

In [14]:
with pool.connection() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT
                CompanyID,
                SUM(CO2_Emissions) AS TotalEmissions
            FROM ESG.EmissionRecords
            GROUP BY CompanyID
        """)

        results = cur.fetchall()

        for row in results:
            print(row)

(3, Decimal('240.10'))
(4, Decimal('850.90'))
(2, Decimal('750.65'))
(1, Decimal('2001.25'))


error connecting in 'pool-1': connection failed: connection to server at "172.198.76.154", port 5432 failed: fe_sendauth: no password supplied
reconnection attempt in pool 'pool-1' failed after 300.0 sec
error connecting in 'pool-1': connection failed: connection to server at "172.198.76.154", port 5432 failed: fe_sendauth: no password supplied
reconnection attempt in pool 'pool-1' failed after 300.0 sec
error connecting in 'pool-1': connection failed: connection to server at "172.198.76.154", port 5432 failed: fe_sendauth: no password supplied
reconnection attempt in pool 'pool-1' failed after 300.0 sec
error connecting in 'pool-1': connection failed: connection to server at "172.198.76.154", port 5432 failed: fe_sendauth: no password supplied
reconnection attempt in pool 'pool-1' failed after 300.0 sec
error connecting in 'pool-2': connection failed: connection to server at "172.198.76.154", port 5432 failed: fe_sendauth: no password supplied
error connecting in 'pool-2': connection 

### Get Pool Statistics

In [15]:
print(
    pool.get_stats()
)

{'requests_num': 5, 'requests_queued': 1, 'connections_num': 4, 'connections_ms': 24654, 'requests_wait_ms': 6889, 'usage_ms': 1343, 'pool_min': 4, 'pool_max': 4, 'pool_size': 4, 'pool_available': 4, 'requests_waiting': 0}
